# Notebook 2: py-TranspaceR Tutorial

Python tutorial for spatial transcriptomics analysis.

In [ ]:
import transspacer as ts
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print(f"py-TranspaceR v{ts.__version__}")

## Load data

In [ ]:
expr = pd.read_csv("D:/test/TranspaceR-main/TranspaceR-main/Example_data/Expression_file_example.csv.gz", index_col=0)
meta = pd.read_csv("D:/test/TranspaceR-main/TranspaceR-main/Example_data/Meta_data_example.csv", index_col=0)
print(f"Cells: {expr.shape[0]}, Genes: {expr.shape[1]}")

## Cell-size normalisation

In [ ]:
normed = ts.c_normalisation(expr.values.astype(float), meta["Area"].values)
print(f"Normalised shape: {normed.shape}")

## Quality control

In [ ]:
threshold = ts.otsu_thresholding(np.log10(expr.values.sum(axis=1) + 1))
print(f"Otsu threshold: {10**threshold:.0f} molecules")

## Variogram analysis

In [ ]:
np.random.seed(42)
idx = np.random.choice(expr.shape[0], 1000, replace=False)
result = ts.compute_variogram(normed[idx, :5], meta["cell_centroid_x"].values[idx], meta["cell_centroid_y"].values[idx])
print(f"Selected: {result['Selected_genes']}")

## Geary C

In [ ]:
coords = np.column_stack([meta["cell_centroid_x"].values[idx], meta["cell_centroid_y"].values[idx]])
gc = ts.geary_c_score(normed[idx, :10], coords, pvalue_threshold=0.05)
print(f"Selected: {len(gc['Selected_genes'])} genes")

## Clustering

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=20, random_state=42)
pca_data = pca.fit_transform(normed[idx])
labels = ts.cell_clustering_function(pca_data, K=10)
print(f"Clusters: {len(np.unique(labels))}")

## Visualization

In [ ]:
ts.plot_fov(meta["cell_centroid_x"].values[idx], meta["cell_centroid_y"].values[idx], labels, title="Spatial Clusters")
plt.show()